1. Import Libraries

In [5]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

2. Check Dataset Path

In [6]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    break

/kaggle/input


3. Check Folder Structure

In [9]:
data_dir = "/kaggle/input/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed"
print(os.listdir(data_dir))

['im_Parabasal', 'im_Dyskeratotic', 'im_Metaplastic', 'im_Superficial-Intermediate', 'im_Koilocytotic']


4. PREPROCESSING

a. STEP 1: CLEAN DATASET

In [10]:
import os
import shutil

original_dir = "/kaggle/input/datasets/prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed"
clean_dir = "/kaggle/working/clean_dataset"

classes = [
    "im_Parabasal",
    "im_Dyskeratotic",
    "im_Metaplastic",
    "im_Superficial-Intermediate",
    "im_Koilocytotic"
]

os.makedirs(clean_dir, exist_ok=True)

for cls in classes:
    src_folder = os.path.join(original_dir, cls, cls)  # nested folder
    dst_folder = os.path.join(clean_dir, cls)
    os.makedirs(dst_folder, exist_ok=True)

    for file in os.listdir(src_folder):
        if file.endswith(".bmp"):   # keep only images
            shutil.copy(
                os.path.join(src_folder, file),
                os.path.join(dst_folder, file)
            )

print("✅ Dataset cleaned!")

✅ Dataset cleaned!


b. STEP 2: VERIFY CLEAN DATA

In [11]:
for cls in classes:
    path = os.path.join(clean_dir, cls)
    print(f"{cls}: {len(os.listdir(path))} images")

im_Parabasal: 108 images
im_Dyskeratotic: 223 images
im_Metaplastic: 271 images
im_Superficial-Intermediate: 126 images
im_Koilocytotic: 238 images


c. 3. Preprocessing + Augmentation

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

# TRAIN (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True
)

# VALIDATION (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

d. 4. Load Dataset

In [16]:
train_data = train_datagen.flow_from_directory(
    clean_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = val_datagen.flow_from_directory(
    clean_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 775 images belonging to 5 classes.
Found 191 images belonging to 5 classes.


e. 5. Verify Everything

In [17]:
# Check class mapping
print(train_data.class_indices)

# Check shapes
images, labels = next(train_data)
print(images.shape)   # (32, 224, 224, 3)
print(labels.shape)   # (32, 5)

{'im_Dyskeratotic': 0, 'im_Koilocytotic': 1, 'im_Metaplastic': 2, 'im_Parabasal': 3, 'im_Superficial-Intermediate': 4}
(32, 224, 224, 3)
(32, 5)


In [18]:
print(train_data.class_indices)

{'im_Dyskeratotic': 0, 'im_Koilocytotic': 1, 'im_Metaplastic': 2, 'im_Parabasal': 3, 'im_Superficial-Intermediate': 4}


6. MODEL BUILDING (MULTICLASS) [(EfficientNet + MobileNet)]

STEP 1: Import

In [20]:
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Concatenate, BatchNormalization
from tensorflow.keras.models import Model

STEP 2: Load Pretrained Models

In [21]:
input_shape = (224, 224, 3)

# EfficientNet
eff_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)

# MobileNet
mob_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)

I0000 00:00:1774457701.157206      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


STEP 3: Freeze Base Layers

In [22]:
for layer in eff_model.layers:
    layer.trainable = False

for layer in mob_model.layers:
    layer.trainable = False

STEP 5: Combine Features (HYBRID)

In [24]:
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Concatenate, BatchNormalization, Input
from tensorflow.keras.models import Model

# Input layer
input_layer = Input(shape=(224, 224, 3))

# Load pretrained models with SAME input
eff_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_tensor=input_layer
)

mob_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=input_layer
)

# Freeze layers
for layer in eff_model.layers:
    layer.trainable = False

for layer in mob_model.layers:
    layer.trainable = False

# Feature extraction
eff_features = GlobalAveragePooling2D()(eff_model.output)
mob_features = GlobalAveragePooling2D()(mob_model.output)

# Combine features
combined = Concatenate()([eff_features, mob_features])

# Classification head
x = BatchNormalization()(combined)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(5, activation='softmax')(x)

# Final model
model = Model(inputs=input_layer, outputs=output)

# Summary
model.summary()

/tmp/ipykernel_55/2824897229.py:15: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mob_model = MobileNetV2(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 6,974,696 (26.61 MB)

 Trainable params: 662,021 (2.53 MB)

 Non-trainable params: 6,312,675 (24.08 MB)

In [25]:
print(model.output_shape)

(None, 5)


6. COMPILATION

In [28]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

7. MODEL TRAINING

a. 1. ADD CLASS WEIGHTS

In [29]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_data.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights_dict = dict(enumerate(class_weights))

print(class_weights_dict)

{0: np.float64(0.8659217877094972), 1: np.float64(0.8115183246073299), 2: np.float64(0.7142857142857143), 3: np.float64(1.7816091954022988), 4: np.float64(1.5346534653465347)}


b. 2. ADD CALLBACKS (PREVENT OVERFITTING)

In [31]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)

c. 3. TRAIN THE MODEL

In [32]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1774458257.227274     194 service.cc:152] XLA service 0x7eb234002540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774458257.227319     194 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774458260.359534     194 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774458273.534544     194 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


25/25 ━━━━━━━━━━━━━━━━━━━━ 80s 2s/step - accuracy: 0.3955 - loss: 1.7494 - val_accuracy: 0.7382 - val_loss: 0.8862 - learning_rate: 0.0010
Epoch 2/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 559ms/step - accuracy: 0.7110 - loss: 0.8164 - val_accuracy: 0.7382 - val_loss: 0.9158 - learning_rate: 0.0010
Epoch 3/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 559ms/step - accuracy: 0.7196 - loss: 0.9147 - val_accuracy: 0.7696 - val_loss: 0.8025 - learning_rate: 0.0010
Epoch 4/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 547ms/step - accuracy: 0.7706 - loss: 0.7796 - val_accuracy: 0.7696 - val_loss: 0.8642 - learning_rate: 0.0010
Epoch 5/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 13s 539ms/step - accuracy: 0.7794 - loss: 0.6341 - val_accuracy: 0.7749 - val_loss: 0.8766 - learning_rate: 0.0010
Epoch 6/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 555ms/step - accuracy: 0.8364 - loss: 0.4615 - val_accuracy: 0.7801 - val_loss: 0.8316 - learning_rate: 3.0000e-04


In [33]:
 for layer in model.layers[-30:]:
    layer.trainable = True

In [34]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    class_weight=class_weights_dict
)

Epoch 1/5


2026-03-25 17:16:50.398281: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 17:16:50.595524: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


25/25 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.6380 - loss: 1.2460 - val_accuracy: 0.7696 - val_loss: 0.9480
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 547ms/step - accuracy: 0.7027 - loss: 0.9521 - val_accuracy: 0.7487 - val_loss: 1.1145
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 545ms/step - accuracy: 0.7177 - loss: 0.7503 - val_accuracy: 0.7016 - val_loss: 1.2905
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 561ms/step - accuracy: 0.7052 - loss: 0.9269 - val_accuracy: 0.6754 - val_loss: 1.4234
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 545ms/step - accuracy: 0.7023 - loss: 0.7809 - val_accuracy: 0.6649 - val_loss: 1.6154


In [36]:
for layer in model.layers[-10:]:   # not 30 ❗
    layer.trainable = True

In [37]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),  # lower!
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [38]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=3,
    class_weight=class_weights_dict
)

Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 0.7526 - loss: 0.6723 - val_accuracy: 0.6492 - val_loss: 1.7369
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 556ms/step - accuracy: 0.6929 - loss: 0.8537 - val_accuracy: 0.6283 - val_loss: 1.8481
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 553ms/step - accuracy: 0.7060 - loss: 0.8873 - val_accuracy: 0.6073 - val_loss: 1.9398


In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils import class_weight
import numpy as np

# 1. Calculate Class Weights to handle dataset imbalance
# (Essential for classes like im_Parabasal which have fewer samples)
labels = train_data.classes
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights_dict = dict(enumerate(weights))

# 2. Define Callbacks for "Perfect Accuracy"
# ReduceLROnPlateau: Drops learning rate when validation loss stalls
# EarlyStopping: Stops training when the model starts overfitting
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_cerviai_model.h5', monitor='val_accuracy', save_best_only=True)
]

# 3. Phase 1: Training the Classification Head (Frozen Bases)
print("Starting Phase 1: Training Classification Head...")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

history_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 4. Phase 2: Progressive Fine-Tuning (Unfreezing Top Layers)
# We unfreeze the top layers of both models to specialize them for cervical cells
print("\nStarting Phase 2: Fine-Tuning Top Layers...")

# Unfreeze the last 20 layers of the base models
for layer in eff_model.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

for layer in mob_model.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

# Re-compile with a much lower learning rate to avoid destroying pretrained weights
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

history_phase2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

print("\n✅ Training Complete. Best model saved as 'best_cerviai_model.h5'")

Starting Phase 1: Training Classification Head...
Epoch 1/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6501 - loss: 1.5131

25/25 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.6496 - loss: 1.5183 - val_accuracy: 0.5969 - val_loss: 5.5755 - learning_rate: 0.0010
Epoch 2/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 555ms/step - accuracy: 0.7231 - loss: 1.1135 - val_accuracy: 0.5550 - val_loss: 8.2823 - learning_rate: 0.0010
Epoch 3/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 550ms/step - accuracy: 0.7966 - loss: 0.7290 - val_accuracy: 0.5550 - val_loss: 10.4877 - learning_rate: 0.0010
Epoch 4/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 485ms/step - accuracy: 0.7875 - loss: 0.7552

25/25 ━━━━━━━━━━━━━━━━━━━━ 15s 582ms/step - accuracy: 0.7883 - loss: 0.7532 - val_accuracy: 0.7592 - val_loss: 4.6021 - learning_rate: 0.0010
Epoch 5/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.8007 - loss: 0.7391

25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 574ms/step - accuracy: 0.8009 - loss: 0.7418 - val_accuracy: 0.7644 - val_loss: 6.8034 - learning_rate: 0.0010
Epoch 6/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 547ms/step - accuracy: 0.8467 - loss: 0.5560 - val_accuracy: 0.4764 - val_loss: 18.0133 - learning_rate: 0.0010
Epoch 7/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - accuracy: 0.8275 - loss: 0.7933
Epoch 7: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 556ms/step - accuracy: 0.8274 - loss: 0.7913 - val_accuracy: 0.6073 - val_loss: 12.8365 - learning_rate: 0.0010
Epoch 8/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 542ms/step - accuracy: 0.8407 - loss: 0.4741 - val_accuracy: 0.5969 - val_loss: 12.8255 - learning_rate: 2.0000e-04
Epoch 9/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 555ms/step - accuracy: 0.8947 - loss: 0.3361 - val_accuracy: 0.5550 - val_loss: 14.1771 - learning_rate: 2.0000e-04
Epoch 9: early stopping
Restoring model weights from the end of the best epoch: 4.

2026-03-25 17:31:54.912219: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-25 17:31:55.120021: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8515 - loss: 0.6165